In [ ]:
# CCX PHASE1 shard 01 — CONFIG (Plan Sec 8)
import os, json
TAG='phase1'
SHARD_ID=1
OUT='/content/ccx_phase1_shard01.csv'
CONFIG_JSON='{"jobs": [{"source": "uniform", "batch": 12, "n": 10000, "seed": 12}, {"source": "uniform", "batch": 13, "n": 10000, "seed": 13}, {"source": "uniform", "batch": 14, "n": 10000, "seed": 14}, {"source": "uniform", "batch": 15, "n": 10000, "seed": 15}, {"source": "uniform", "batch": 16, "n": 10000, "seed": 16}, {"source": "uniform", "batch": 17, "n": 10000, "seed": 17}, {"source": "uniform", "batch": 18, "n": 10000, "seed": 18}, {"source": "uniform", "batch": 19, "n": 10000, "seed": 19}, {"source": "uniform", "batch": 20, "n": 10000, "seed": 20}, {"source": "uniform", "batch": 21, "n": 10000, "seed": 21}, {"source": "uniform", "batch": 22, "n": 10000, "seed": 22}, {"source": "uniform", "batch": 23, "n": 10000, "seed": 23}]}'
CONFIG=json.loads(CONFIG_JSON)
print(TAG, 'shard', SHARD_ID, 'groups', len(CONFIG.get('groups', CONFIG.get('jobs', []))))


In [ ]:
# ---- setup: clone pinned repo + deps (thin) ----
import os, sys, subprocess, pathlib, json, time, hashlib

REPO_DIR = "/tmp/ccx"
GIT_URL = "https://github.com/hugogobato/ccx-contextual-confounding.git"
GIT_SHA = "d2dd265cef5ef171684997e8115126d75a2dd4cd"

# pip deps (numpy/scipy/pandas/matplotlib already on Colab, ensure versions)
# keep install light; inflation not needed for these jobs but harmless
try:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "scipy==1.17.1", "pandas==3.0.1"])
except Exception as e:
    print("pip install warning:", e)

# clone or reuse
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print(f"cloning {GIT_URL} @ {GIT_SHA[:7]} -> {REPO_DIR}")
    # try anonymous clone first (works if repo public); if private, try token from Colab secrets
    tok = None
    try:
        from google.colab import userdata
        tok = userdata.get("CCX_GH_TOKEN")
    except Exception:
        tok = os.environ.get("CCX_GH_TOKEN")
    url = GIT_URL
    if tok:
        # inject token: https://<token>@github.com/...
        url = GIT_URL.replace("https://", f"https://{tok}@")
        print("using GH token from secrets/env")
    subprocess.check_call(["git", "clone", url, REPO_DIR])
else:
    print("repo already cloned, fetching")
    subprocess.check_call(["git", "-C", REPO_DIR, "fetch", "--all", "-q"])

subprocess.check_call(["git", "-C", REPO_DIR, "checkout", GIT_SHA, "-q"])
# ensure src on path and ROOT env for imports
if REPO_DIR + "/src" not in sys.path:
    sys.path.insert(0, REPO_DIR + "/src")
os.chdir(REPO_DIR)
print("checked out", subprocess.check_output(["git","-C",REPO_DIR,"rev-parse","--short","HEAD"], text=True).strip())
# verify
import numpy, scipy, pandas
print("numpy", numpy.__version__, "scipy", scipy.__version__, "pandas", pandas.__version__)
CODE_HASH = GIT_SHA[:16]


In [ ]:

# ---- Phase 1 enumeration driver (sharded) ----
import numpy as np, pandas as pd
from pathlib import Path
from models import build_iv_A, iv_vertices, discover_facets
from witnesses import slack_and_feasible, cf1_soft, degree_signed, kl_em_batch, iv_cf1_hard, maximal_support_lp, iv_inflation2_feasible
from dgps import uniform_conditionals, boundary_batch, sparse_conditionals
import json as _json

# load facets (generate if missing)
FACETS_CACHE = Path("results/phase1_enumeration/facets_iv.npz")
if FACETS_CACHE.exists():
    z = np.load(FACETS_CACHE)
    H_fac, b_fac = z["H"], z["b"]
else:
    print("facets cache missing, generating via discover_facets...")
    H_fac, b_fac = discover_facets(build_iv_A(), seed=0)
    FACETS_CACHE.parent.mkdir(parents=True, exist_ok=True)
    np.savez(FACETS_CACHE, H=H_fac, b=b_fac)
    print("facets generated:", H_fac.shape)

A16 = build_iv_A()
Af = A16.astype(float)
TOL_BORDER_LO, TOL_BORDER_HI = 1e-9, 1e-7

def process_batch_inline(E, source, batch_label, seed):
    n = len(E)
    fac_ok = np.all(H_fac.astype(float) @ E.T <= b_fac[:, None].astype(float) + 1e-9, axis=0)
    kl_vals, _ = kl_em_batch(Af, E)
    rows=[]
    for i in range(n):
        e=E[i]
        slack, feas = slack_and_feasible(Af, e)
        border = TOL_BORDER_LO < slack < TOL_BORDER_HI
        row={"instance_id": f"{source}_{batch_label}_{i}", "source": source, "batch": batch_label, "seed": seed,
             **{f"e_{j}": float(e[j]) for j in range(8)},
             "lp_feasible": bool(feas), "facets_ok": bool(fac_ok[i]), "agree_c1a": bool(feas==bool(fac_ok[i])),
             "slack_l1": float(slack), "borderline_band": bool(border),
             "cf1_soft": cf1_soft(Af, e), "degree_signed": degree_signed(Af, e),
             "kl_contextuality": float(kl_vals[i]), "inflation2_feasible": np.nan}
        if not feas and not border:
            row["cf1_hard"]=iv_cf1_hard(e, A16)
            m0=maximal_support_lp(A16[0:4], e[0:4]>0)
            m1=maximal_support_lp(A16[4:8], e[4:8]>0)
            row["maximally_contextual"]=bool(max(m0,m1)<=1e-9)
        else:
            row["cf1_hard"]=np.nan
            row["maximally_contextual"]=False
        rows.append(row)
    return rows

# resume: check which batches already in OUT
rows_all=[]
done_batches=set()
if os.path.exists(OUT):
    try:
        prev=pd.read_csv(OUT)
        if "batch" in prev.columns and "source" in prev.columns:
            done_batches=set(zip(prev["source"], prev["batch"].astype(int)))
        rows_all=prev.to_dict("records")
        print(f"resume: {len(done_batches)} batches, {len(rows_all)} rows already in OUT")
    except Exception as e:
        print("resume read failed", e)

jobs=CONFIG["jobs"]
todo=[j for j in jobs if (j["source"], j["batch"]) not in done_batches]
print(f"jobs total {len(jobs)} todo {len(todo)}")

for idx, job in enumerate(todo):
    src=job["source"]; batch=job["batch"]; seed=job["seed"]; n=job["n"]
    print(f"[{idx+1}/{len(todo)}] {src} batch={batch} n={n} seed={seed}", flush=True)
    t0=time.time()
    if src=="uniform":
        rng=np.random.default_rng(seed)
        E=uniform_conditionals(rng, n)
    elif src=="sparse":
        rng=np.random.default_rng(seed)
        # sparse: use dgps sparse_conditionals directly
        E=sparse_conditionals(rng, n)
    else:  # boundary
        E=boundary_batch(A16, H_fac, b_fac, n, seed)
    rows=process_batch_inline(E, src, batch, seed)
    rows_all.extend(rows)
    # incremental save every job (small jobs, cheap)
    pd.DataFrame(rows_all).to_csv(OUT, index=False)
    print(f"  -> {len(rows)} rows, {time.time()-t0:.1f}s, total {len(rows_all)}", flush=True)

# final manifest
manifest={"tag": TAG, "shard_id": SHARD_ID, "code_hash": CODE_HASH, "jobs": len(jobs), "rows": len(rows_all), "git_sha": GIT_SHA}
mpath=f"/content/ccx_{TAG}_manifest_shard{SHARD_ID:02d}.json"
with open(mpath,"w") as fh: json.dump(manifest,fh,indent=2)
print("MANIFEST", json.dumps(manifest))
try:
    from google.colab import files
    files.download(OUT); files.download(mpath)
    print("downloaded", OUT)
except Exception as e:
    print("(download skipped)", e)
